In [4]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent

# agent = create_agent(model="deepseek-chat", tools=[get_weather])
from langchain.chat_models import init_chat_model
import os
base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")
model = init_chat_model(
    model="qwen3-max-2026-01-23",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)
agent = create_agent(model=model, tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
    ]
})

print(response)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='041aeeae-63d7-40a2-9257-eb5a7325faa9'), HumanMessage(content='你好，我是虎哥.', additional_kwargs={}, response_metadata={}, id='e43b617b-7835-4066-beb0-71e41b438b11'), AIMessage(content='你好，虎哥，很高兴认识你.', additional_kwargs={}, response_metadata={}, id='f3d45ddd-5aac-4089-ad2f-fcdf7ba81b99', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='1c531d4a-b3df-4108-85a6-edafc80388b1'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 304, 'total_tokens': 325, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3-max-2026-01-23', 'system_fingerprint': None, 'id': 'chatcmpl-5790bb21-6cb7-95e9-92ae-f982db174def', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dcfe3-9fa4-7

In [6]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎哥.
================================== Ai Message ==================================

你好，虎哥，很高兴认识你.
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_cea4dd3071514c64884f112a)
 Call ID: call_cea4dd3071514c64884f112a
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

Current weather in 北京 is sunny
================================== Ai Message ==================================

北京今天天气晴朗，适合外出活动！


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [7]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [8]:
# 创建Agent
agent = create_agent(model=model)

In [9]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [10]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


这张图片描绘了一个非常温馨、治愈的海滩场景。以下是详细的内容描述：

**主要人物与动作：**
*   **年轻女子：** 一位长发年轻女子侧身坐在沙滩上。她身穿一件蓝白格纹的长袖衬衫和深色长裤（裤脚卷起）。她面带灿烂的笑容，神情非常放松和愉悦。
*   **狗：** 在她对面坐着一只浅金色的大型犬（看起来像金毛寻回犬或拉布拉多）。狗狗温顺地抬起一只前爪，与女子伸出的手掌轻轻相触，看起来像是在玩“握手”或击掌的游戏。
*   **互动：** 两者之间的互动充满了信任和亲密感，展现了人与宠物之间深厚的友谊。

**环境与背景：**
*   **地点：** 场景设定在宽阔的沙滩上，沙子看起来细腻柔软，上面有一些自然的纹理和脚印。
*   **背景：** 远处是平静的大海，可以看到轻微的海浪拍打着海岸。天空非常明亮，几乎与海平面融为一体。
*   **光影：** 图片的光线非常柔和且温暖，看起来像是日落时分（黄金时刻）。阳光从画面的右侧照射过来，形成了一种逆光的效果，给女子的头发和侧脸镀上了一层金色的光晕，营造出一种梦幻、宁静的氛围。

**细节：**
*   狗狗身上佩戴着一条带有彩色图案的胸背带（harness）。
*   一条红色的牵引绳散落在女子脚边的沙地上，表明他们是在散步或玩耍途中停下来休息。

总的来说，这是一张充满阳光、快乐和宁静氛围的照片，捕捉了人与动物和谐相处的美好瞬间。

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [19]:
!uv add ipywidgets

Resolved 195 packages in 1ms
Checked 191 packages in 5ms


然后，我们创建一个上传组件，用于模拟图片上传。


In [15]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [16]:
print(uploader.value)

({'name': '鼠鼠得吃.jpg', 'type': 'image/jpeg', 'size': 160054, 'content': <memory at 0x0000022835721480>, 'last_modified': datetime.datetime(2025, 11, 2, 7, 37, 54, 994000, tzinfo=datetime.timezone.utc)},)


In [17]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [18]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])


for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


这就有点尴尬了……这张图片里其实**并没有展示任何现实中的城市**。

这看起来是一张典型的网络**表情包（Meme）**，或者说是“抽象图”。我们可以拆解一下图里的元素，你就明白为什么找不到城市了：

1.  **核心文字：“得吃！”**
    这是网络用语，意思是“必须吃”、“一定要拿下”或者“这个好东西归我了”。通常用来表达看到美食、钱财（比如图里的钻石）或者心仪物品时那种贪婪、兴奋或者势在必得的心情。

2.  **主角：一只魔性的猫**
    这只猫的脸被拉伸变形了，表情看起来非常亢奋、甚至有点疯狂，配合“得吃！”的文字，增强了那种“我要把这个东西吞掉”的喜剧效果。

3.  **背景元素：**
    *   **左下角：** 一颗巨大的钻石（代表财富或珍贵的东西）。
    *   **后方和右侧：** 看起来像是电子游戏里的素材。那个红色的箱子（上面隐约有 INDUSTRIAL 字样）和右边的屏幕，非常有《Garry's Mod》（GMod）或者《半条命2》（Half-Life 2）这种基于Source引擎游戏的风格。

**总结：**
这张图并不是在展示某个城市，而是在表达一种**“看到好东西（钻石），我要把它据为己有/吃掉”**的搞笑情绪。

如果你是在某个特定的游戏群或论坛看到的，它可能是在调侃游戏里的某个道具或者掉落的装备。